In [1]:
import sys
from pathlib import Path

sys.path.append(Path.cwd().parent.parent.as_posix())
import logging

from demo_and_analysis.benchmark.systems.bespoke import BespokeRunner
from pipeline.git_snapshotter import GitSnapshotter
from utils.logging_and_reporting.logger import setup_logging

setup_logging(logging.DEBUG)

out_path = Path.cwd() / "output"
out_path.mkdir(exist_ok=True)

snapshotter = GitSnapshotter(
    cache_repo="git://c01/bespoke_cache.git",
    working_dir=out_path,
    extra_gitignore=[],
)
snapshotter.fetch_snapshots()

bespoke_runner = BespokeRunner(db_engine=None, snapshotter=snapshotter)

In [2]:
# get info from wandb
from demo_and_analysis.plots.utils.wandb_utils import get_wandb_stats

wandb_run = "slgev39o"


summary, hist, config = get_wandb_stats(
    wandb_run,
    skip_cache=False,  # set to True to skip cache and fetch from W&B API
    wandb_run_cache_path=Path("/mnt/labstore/bespoke_olap/wandb_cache"),
)

assert summary is not None, "Failed to retrieve summary from W&B"
assert config is not None, "Failed to retrieve config from W&B"
snapshot_hash: str = summary["code/snapshot_hash"]  # type: ignore
benchmark: str = config["benchmark"]  # type: ignore
query_list: list = config["query_list"].split(",")  # type: ignore


assert isinstance(snapshot_hash, str), "snapshot_hash should be a string"
assert isinstance(benchmark, str), "benchmark should be a string"
assert isinstance(query_list, list), (
    f"query_list should be a list. got: {type(query_list)}"
)

print(f"snapshot_hash: {snapshot_hash}")
print(f"benchmark: {benchmark}")
print(f"query_list: {query_list}")

Loaded wandb data from cache: /mnt/labstore/bespoke_olap/wandb_cache/3a0cffe3df25dfd926a95c743b5107b84c971d50181079c5daa7e61ef90bdb6e.pkl
snapshot_hash: d5bddd96922e76ddceab4eb68fd0e9c9eb157fe6
benchmark: tpch
query_list: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22']


In [6]:
bespoke_runner.restore_snapshot(
    snapshot=snapshot_hash,
    benchmark=benchmark,
    query_list=query_list,
    cache_path=Path("/mnt/labstore/bespoke_olap/cache"),
)

In [8]:
# copy in helper files
api_path = Path.cwd().parent.parent / "pipeline" / "cpp_runner"
files = [
    "builder_api.cpp",
    "builder_api.hpp",
    "loader_api.cpp",
    "loader_api.hpp",
    "loader_utils.cpp",
    "loader_utils.hpp",
    "cpu_affinity.cpp",
    "cpu_affinity.hpp",
]

for file in files:
    src = api_path / file
    dst = snapshotter.working_dir / file

    assert src.exists(), f"Source file {src} does not exist."
    if not dst.exists():
        # copy file
        dst.write_bytes(src.read_bytes())
    else:
        print(f"File {dst} already exists, skipping copy.")


File /home/jwehrstein/bespoke_olap/demo_and_analysis/prepare_code_for_export/output/builder_api.cpp already exists, skipping copy.
File /home/jwehrstein/bespoke_olap/demo_and_analysis/prepare_code_for_export/output/builder_api.hpp already exists, skipping copy.
File /home/jwehrstein/bespoke_olap/demo_and_analysis/prepare_code_for_export/output/loader_api.cpp already exists, skipping copy.
File /home/jwehrstein/bespoke_olap/demo_and_analysis/prepare_code_for_export/output/loader_api.hpp already exists, skipping copy.
File /home/jwehrstein/bespoke_olap/demo_and_analysis/prepare_code_for_export/output/loader_utils.cpp already exists, skipping copy.
File /home/jwehrstein/bespoke_olap/demo_and_analysis/prepare_code_for_export/output/loader_utils.hpp already exists, skipping copy.
File /home/jwehrstein/bespoke_olap/demo_and_analysis/prepare_code_for_export/output/cpu_affinity.cpp already exists, skipping copy.
File /home/jwehrstein/bespoke_olap/demo_and_analysis/prepare_code_for_export/outpu

In [12]:
# strip lines
# // Increment file version to invalidate cache when this file is changed. This is needed because this file is included in the generated code and changes to it should trigger regeneration of all code that includes it.
# // FILE_VERSION: 1

for file in list(snapshotter.working_dir.glob("*.cpp")) + list(
    snapshotter.working_dir.glob("*.hpp")
):
    lines = file.read_text().splitlines()
    lines = [line for line in lines if not line.startswith("// FILE_VERSION")]
    lines = [line for line in lines if not line.startswith("// Increment file version")]

    file.write_text("\n".join(lines))

# remove .out files
for file in snapshotter.working_dir.glob("*.out"):
    file.unlink()

# create a zip file
zip_path = snapshotter.working_dir.parent / f"{snapshot_hash}.zip"

if zip_path.exists():
    print(f"Zip file {zip_path} already exists, skipping creation.")
else:
    import zipfile

    with zipfile.ZipFile(zip_path, "w") as zip_file:
        for file in snapshotter.working_dir.glob("*"):
            if file.name == ".git" or ".git" in file.parts:
                continue
            zip_file.write(file, arcname=file.name)
